# SVEN Colab Environment Check
This notebook verifies that you're running on Google Colab and inspects key runtime details.
import sys, platform, os
is_colab = 'google.colab' in sys.modules
print(f"Running on Colab: {is_colab}")
print(f"Python: {platform.python_version()} | Platform: {platform.platform()}")
print(f"Working dir: {os.getcwd()}")
print(f"CUDA_VISIBLE_DEVICES: {os.environ.get('CUDA_VISIBLE_DEVICES')}")

In [ ]:
# Test GPU Availability
# This cell checks for a CUDA GPU and prints details using torch and nvidia-smi (if available).
import shutil, subprocess, textwrap, json
try:
    import torch
    gpu_available = torch.cuda.is_available()
    print(f"Torch CUDA available: {gpu_available}")
    if gpu_available:
        print(f"CUDA device count: {torch.cuda.device_count()}")
        print(f"Current device: {torch.cuda.current_device()} - {torch.cuda.get_device_name(0)}")
    else:
        print("No GPU detected. In Colab, go to Runtime > Change runtime type > GPU.")
except Exception as e:
    print("Torch not installed yet or error querying GPU:", e)

# nvidia-smi output
if shutil.which('nvidia-smi'):
    print("nvidia-smi:")
    print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total,memory.free','--format=csv,noheader'], text=True))
else:
    print("nvidia-smi not found (likely no GPU runtime).")

In [ ]:
# Test TPU Availability
# Detect TPU presence in Colab and show basic info.
import os
if 'COLAB_TPU_ADDR' in os.environ:
    print('TPU detected at', os.environ['COLAB_TPU_ADDR'])
    try:
        import torch_xla.core.xla_model as xm
        print('XLA device:', xm.xla_device())
    except Exception as e:
        print('torch-xla not available:', e)
else:
    print('No TPU detected. If needed, switch to TPU in Runtime settings.')

In [ ]:
# Mount Google Drive
# This lets you read/write files in your Drive (optional).
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted at /content/drive')
except Exception as e:
    print('Not running in Colab or Drive not available:', e)

In [ ]:
# Install Additional Packages
# Install minimal dependencies for SVEN dry-run + CodeGen 350M (optional).
%pip install --upgrade pip
# Core deps (adjust if your requirements.txt splits):
%pip install torch --extra-index-url https://download.pytorch.org/whl/cu118
%pip install transformers sentencepiece
# (Optional) For YAML config parsing if not already installed:
%pip install pyyaml

# (Optional) Hugging Face login if private models are needed:
# from huggingface_hub import login
# login()  # paste your token

In [ ]:
# Test Basic Python Operations
print('Hello SVEN!')
print('2 + 2 =', 2 + 2)

# Simple tensor test if torch is installed
try:
    import torch
    x = torch.randn(2,3).cuda() if torch.cuda.is_available() else torch.randn(2,3)
    print('Tensor device:', x.device)
    print('Tensor mean:', x.mean().item())
except Exception as e:
    print('Torch test skipped:', e)

# SVEN: Clone repository and set up
This section clones the project repo, checks out the preferred branch, and installs dependencies for a dry-run.

In [ ]:
# Clone repo and install project requirements
import os, subprocess, sys
repo_url = 'https://github.com/rjucr18/cs260-project.git'
repo_dir = 'cs260-project'
if not os.path.exists(repo_dir):
    !git clone $repo_url
else:
    print('Repo already cloned.')
%cd cs260-project

# (Optional) checkout your branch if needed:
branch = 'rohit-model-prefix-docker'
!git fetch origin
!git checkout $branch || echo('Branch checkout failed, staying on default')

# Install project requirements (edit if splits are added later)
!pip install -r requirements.txt || echo('requirements.txt install encountered issues')

print('Setup complete. Current working directory:', os.getcwd())

In [ ]:
# Secure vs Vulnerable Generation Demo
# Uses the scripts/generate_secure_compare.py to produce side-by-side outputs.
import json, os, sys
%cd scripts
prompt = "def connect(user_input):\n    pass"
!python generate_secure_compare.py --prompt "$prompt" --max-length 64
%cd ..

output_path = os.path.join('logs','secure_compare.json')
if os.path.exists(output_path):
    with open(output_path,'r') as f:
        data = json.load(f)
    print('Generation Compare:')
    print(json.dumps(data, indent=2)[:1200])
else:
    print('secure_compare.json not found. Check script run output above.')

In [ ]:
# Training Dry-Run (timing-only)
# Runs a short loop to produce logs/metrics.json without heavy compute.
!python train.py --config configs/python.yaml --steps 10 --timing-only

import json, os
metrics_path = os.path.join('logs','metrics.json')
if os.path.exists(metrics_path):
    with open(metrics_path,'r') as f:
        metrics = json.load(f)
    print('Trainer metrics:')
    print(json.dumps(metrics, indent=2))
else:
    print('metrics.json not found. Check train.py output above.')

In [ ]:
# Notes & Troubleshooting
print("""Troubleshooting Tips:\n- GPU OOM: Switch to smaller model (350M), reduce --max-length.\n- transformers import error: Re-run pip install cell.\n- Hugging Face rate limits: Create an account and use a token (login code in earlier cell).\n- CodeQL: Heavy and not ideal for Colab ephemeral sessions; run locally or in a dedicated VM.\n- logs/ directory missing: Scripts create it automatically; ensure repo root (use %cd cs260-project).\n- metrics.json empty: Ensure --timing-only and --steps flags executed; check train.py output.\n""")